In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests

In [4]:
from langchain.agents import create_agent

In [6]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [7]:
search_tool = TavilySearch(max_results=1)


In [9]:
@tool
def get_weather(city: str) -> str:
    """Fetch current weather information for a city"""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "WEATHERSTACK_API_KEY is not set"

    url = (
        f"http://api.weatherstack.com/current"
        f"?access_key={api_key}"
        f"&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [8]:
result = search_tool.invoke("Whats the latest updates on Grammy")
result

{'query': 'Whats the latest updates on Grammy',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.grammy.com/news',
   'title': 'Archives: News - Grammy',
   'content': 'Grammy\n\n# Grammy News & Updates\n\nAll the latest news and updates from across the Grammy organization.\n\n## Latest News\n\nA graphic promoting the Online Entry Process (OEP) for the 2027 Grammys. The graphic features a hand on the left holding a Grammy Award statue facing left and the words \n\nGraphic: Courtesy of the Recording Academy\n\n## How To Submit Your Music For The 2027 Grammys: Deadlines, Grammy Nominations, Grammy Voting Process & More\n\n## Ready to Reach Out? We’re Ready to Listen.\n\n## More',
   'score': 0.7570233,
   'raw_content': None,
   'id': 'd7fc30-00'}],
 'response_time': 0.0,
 'request_id': 'af8a8d67-af2d-4b8e-a901-5083456fa83c'}

In [10]:
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"]
)

In [11]:
response = llm.invoke("What  is curreny year")
response

AIMessage(content='The current year is **2026**.', additional_kwargs={'reasoning_content': 'The user asks: "What  is curreny year". They probably want the current year. The assistant should answer "2026". Also maybe mention "The current year is 2026." Should be simple. The user wrote "curreny" maybe a typo. Just answer.'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 77, 'total_tokens': 153, 'completion_time': 0.084922888, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.006446451, 'prompt_tokens_details': None, 'queue_time': 0.316021476, 'total_time': 0.091369339}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0804d-9c3f-7980-b267-5850e4b89b4d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 76, 'total_tokens': 153, 'output_token_details': {'rea

In [12]:
tools = [search_tool, get_weather]

In [13]:
agent = create_agent(
    model = llm,
    tools = tools

)

In [14]:
response = agent.invoke({
    "messages":[
        {"role": "user", "content": "Find the current weatherin ongole"}
    ]
})

In [15]:
print(response["messages"] [-1].content)

**Current weather in Ongole (as of 2026‑09‑08 14:49 IST)**  

| Item | Value |
|------|-------|
| **Temperature** | 37 °C (98.6 °F) |
| **Feels‑like** | 36.6 °C (97.9 °F) |
| **Condition** | Overcast |
| **Humidity** | 35 % |
| **Wind** | 18.4 kph (11.4 mph) from the NW (306°) |
| **Pressure** | 1005 mb (29.67 in) |
| **Precipitation** | 0 mm (0 in) – no rain expected |
| **Cloud cover** | 100 % |
| **Visibility** | 10 km (6 mi) |
| **UV Index** | 2.5 |

*Source: WeatherAPI.com – current weather data for Ongole, Andhra Pradesh, India (retrieved via a real‑time search)*【tavily_search】.


In [16]:
for msg in response ["messages"]:
    print(msg.content)

Find the current weatherin ongole

WEATHERSTACK_API_KEY is not set

{"query": "current weather in Ongole", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "Weather in Ongole", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'Ongole', 'region': 'Andhra Pradesh', 'country': 'India', 'lat': 15.5, 'lon': 80.05, 'tz_id': 'Asia/Kolkata', 'localtime_epoch': 1788859144, 'localtime': '2026-09-08 14:49'}, 'current': {'last_updated_epoch': 1788858000, 'last_updated': '2026-09-08 14:30', 'temp_c': 37.0, 'temp_f': 98.6, 'is_day': 1, 'condition': {'text': 'Overcast', 'icon': '//cdn.weatherapi.com/weather/64x64/day/122.png', 'code': 1009}, 'wind_mph': 11.4, 'wind_kph': 18.4, 'wind_degree': 306, 'wind_dir': 'NW', 'pressure_mb': 1005.0, 'pressure_in': 29.67, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 35, 'cloud': 100, 'feelslike_c': 36.6, 'feelslike_f': 97.9, 'windchill_c': 37.0, 'windchill_f': 98.6, 'heatindex_c': 40.0, 'heatindex_f': 104.0